# Compute and plot atmospheric forecast-period bias metrics

Read model ensembles and references only for missing/forced products, cache metrics, and plot the cached maps.

In [ ]:
# =============================================================================
# TOP-LEVEL PATH PARAMETERS
# =============================================================================
from pathlib import Path
import sys

INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_out")
WORKFLOW_NAME = "fcst_atm_bias"

DIAGNOSTIC_DIR = WORK_DIR / "diagnostic"
if not DIAGNOSTIC_DIR.is_dir():
    raise FileNotFoundError(f"Diagnostic work directory not found: {DIAGNOSTIC_DIR}")
if str(DIAGNOSTIC_DIR) not in sys.path:
    sys.path.insert(0, str(DIAGNOSTIC_DIR))

from configs.output_paths import workflow_output_dirs
WORKFLOW_DATA_DIR, WORKFLOW_FIGURE_DIR = workflow_output_dirs(
    WORKFLOW_NAME, output_root=DIAGNOSTIC_OUTPUT_ROOT
)


## Notes: available variables, metrics, and periods

This notebook handles the **forecast period** only (`run_segment="fc"`).
It is intentionally separate from the DA-analysis-period workflow so their experiment
sets, periods, caches, and figures cannot be mixed.

Available atmospheric variables:

| Variable | Reference | Units |
|---|---|---|
| `FLUT` | CERES-OAFlux | W m-2 |
| `PRECT` | GPCP | mm day-1 |
| `TMQ` | ERA5 | kg m-2 |
| `TS` | ERA5 | degC |
| `TREFHT` | ERA5 | degC |
| `PSL` | ERA5 | hPa |

Available metrics:

| Metric | Meaning |
|---|---|
| `bias` | Ensemble-mean model minus reference, averaged over the selected period |
| `rmse` | Root-mean-square member error over ensemble and time |
| `spread` | Ensemble standard deviation averaged over time |
| `crps` | Continuous ranked probability score averaged over time |

Available experiment groups:

- `S0`: analysis period `201112`; forecast period `201201–201202`.
- `S1`: analysis period `201205`; forecast period `201206–201207`.

Only experiments with a configured `fc` run are available. Use
`available_experiments(SEASON, RUN_SEGMENT)` after the imports to list them.
Each experiment/variable/period has a request-keyed NetCDF cache. Existing
products are reused unless `force_compute=True`. Set `N_BOOTSTRAP=0` to skip
confidence intervals; bias hatching is drawn only when cached intervals exist.


## Selection

In [ ]:
REFERENCE_DATA_ROOT = INPUT_DATA_ROOT / "reference"
CACHE_DIR = WORKFLOW_DATA_DIR / "bias_metrics"
FIGURE_DIR = WORKFLOW_FIGURE_DIR / "bias_metrics"

RUN_SEGMENT = "fc"  # Fixed by this workflow: da=analysis, fc=forecast.
COMPONENT = "atm"
SEASON = "S0"
EXPERIMENTS = ["CTRL10-S0", "CAPT10-S0", "DART20-S0", "DART40-S0"]
SELECTED_VARIABLES = ["FLUT", "PRECT", "TMQ", "TS", "TREFHT", "PSL"]
METRICS_TO_PLOT = ["bias", "rmse", "spread", "crps"]

LAT_RANGE = (-90, 90)
LON_RANGE = (-180, 180)
N_BOOTSTRAP = 100
RANDOM_SEED = 42
MAX_MEMBERS = None  # Set a small integer only for a quick test.

force_compute = False
RUN_WORKFLOW = False  # Review selections, then set True.


## Imports and availability

In [ ]:
import matplotlib.pyplot as plt
import xarray as xr

from util.bias_metrics import (
    ATM_VARIABLES,
    LND_VARIABLES,
    METRICS,
    available_experiments,
    cache_bias_metrics,
    cache_path,
    experiment_registry,
    plot_metric_maps,
    read_model_ensemble,
    read_reference,
)

print("Available experiments:", available_experiments(SEASON, RUN_SEGMENT))


## 1. Read, process, and cache

In [ ]:
if RUN_WORKFLOW:
    variable_registry = ATM_VARIABLES if COMPONENT == "atm" else LND_VARIABLES
    unknown_variables = sorted(set(SELECTED_VARIABLES) - set(variable_registry))
    if unknown_variables:
        raise ValueError(f"Unsupported {COMPONENT} variables: {unknown_variables}")

    registry = experiment_registry(INPUT_DATA_ROOT, SEASON, RUN_SEGMENT)
    unknown_experiments = sorted(set(EXPERIMENTS) - set(registry))
    if unknown_experiments:
        raise ValueError(
            f"Experiments unavailable for {SEASON}/{RUN_SEGMENT}: "
            f"{unknown_experiments}; choose from {list(registry)}"
        )

    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    for variable in SELECTED_VARIABLES:
        references = {}
        for experiment in EXPERIMENTS:
            config = registry[experiment]
            period = config["period"]
            output = cache_path(CACHE_DIR, experiment, variable, period)
            if output.is_file() and not force_compute:
                print(f"Reusing {output.name}")
                continue

            if period not in references:
                references[period] = read_reference(
                    REFERENCE_DATA_ROOT, COMPONENT, variable, period,
                    LAT_RANGE, LON_RANGE,
                )
            model, _ = read_model_ensemble(
                INPUT_DATA_ROOT, experiment, SEASON, RUN_SEGMENT,
                COMPONENT, variable, period=period,
                lat_range=LAT_RANGE, lon_range=LON_RANGE,
                max_members=MAX_MEMBERS,
            )
            try:
                dataset = cache_bias_metrics(
                    output,
                    model=model,
                    reference=references[period],
                    force_compute=force_compute,
                    experiment=experiment,
                    variable=variable,
                    component=COMPONENT,
                    run_segment=RUN_SEGMENT,
                    season=SEASON,
                    period=period,
                    n_bootstrap=N_BOOTSTRAP,
                    random_seed=RANDOM_SEED,
                )
                dataset.close()
            finally:
                model.close()
        for reference in references.values():
            reference.close()


## 2. Plot cached metrics

In [ ]:
if RUN_WORKFLOW:
    registry = experiment_registry(INPUT_DATA_ROOT, SEASON, RUN_SEGMENT)
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    label_map = {name: registry[name]["label"] for name in EXPERIMENTS}
    for variable in SELECTED_VARIABLES:
        paths = {
            experiment: cache_path(
                CACHE_DIR, experiment, variable, registry[experiment]["period"]
            )
            for experiment in EXPERIMENTS
        }
        missing = [path for path in paths.values() if not path.is_file()]
        if missing:
            raise FileNotFoundError(
                "Missing cached bias metrics:\n" +
                "\n".join(str(path) for path in missing)
            )
        cached = {
            experiment: xr.load_dataset(path)
            for experiment, path in paths.items()
        }
        try:
            for metric in METRICS_TO_PLOT:
                if metric not in METRICS:
                    raise ValueError(f"Unknown metric: {metric}")
                figure = plot_metric_maps(
                    cached, metric, variable, label_map=label_map,
                    title=(
                        f"{variable} {metric.upper()} | {SEASON} "
                        f"{RUN_SEGMENT.upper()}"
                    ),
                )
                figure.savefig(
                    FIGURE_DIR / f"{variable}_{metric}_{SEASON}_{RUN_SEGMENT}.png",
                    dpi=200, bbox_inches="tight",
                )
                plt.close(figure)
        finally:
            for dataset in cached.values():
                dataset.close()
